# 🚗 Domácí úkol (Session 2): Random Forest – Predikce cen automobilů (Regrese)
**Dataset:** `car_data.csv` (19 237 inzerátů, 18 atributů)  
**Cíl úlohy:** Konstrukce a optimalizace modelu náhodného lesa (`RandomForestRegressor`) pro predikci prodejní ceny automobilu (`price`), včetně detailní inspekce přes `ydata_profiling`, čištění zbytečných identifikátorů, nápravy poškozených záznamů (Excel auto-format u `doors`), ošetření anomálních odlehlých hodnot, One-Hot kódování, rozdělení 70:30, hyperparametrického ladění s optimalizací na MSE a vyhodnocení přes MAE a RMSE.

---
### 📋 Kroky zadání:
1. **Načtení dat:** Stáhněte soubor `car_data.csv` a načtěte jej pomocí knihovny Pandas.
2. **Import tříd:** Importujte potřebné nástroje z knihovny `scikit-learn`.
3. **Data Profiling report:** Zobrazte souhrnný report kvality dat pomocí knihovny `ydata_profiling`:
   ```python
   !pip install ydata_profiling
   import ydata_profiling as ydp
   cars_report = ydp.ProfileReport(cars_df)
   cars_report
   ```
4. **Čištění dat:** Odstraňte nepotřebné sloupce (`Unnamed: 0`, `id`, `model`) a opravte poškozené kategorické hodnoty (zejména Excel auto-konverzi u `doors`: `'04-May'` $\to$ `'4-5'`, `'02-Mar'` $\to$ `'2-3'`).
5. **One-Hot Encoding:** Zakódujte kategorické atributy na binární příznaky.
6. **Train-Test Split:** Rozdělte data na trénovací a testovací množinu v poměru 70:30 (`random_state=42`).
7. **Instance modelu:** Inicializujte `RandomForestRegressor` s parametry `random_state` a `n_jobs`.
8. **Mřížka hyperparametrů:** Definujte slovník `params` pro klíče `max_depth`, `min_samples_leaf` a `n_estimators`.
9. **Optimalizace hyperparametrů:** Pomocí `GridSearchCV` nalezněte nejlepší kombinaci s optimalizovanou metrikou `mean_squared_error` (`scoring='neg_mean_squared_error'`).
10. **Dotrénování nejlepšího modelu:** Natrénujte nejlepší model na `X_train, y_train`.
11. **Predikce:** Proveďte odhady na testovací sadě `X_test`.
12. **Vyhodnocení:** Spočítejte průměrnou absolutní chybu (**MAE**) a odmocninu průměrné čtvercové chyby (**RMSE**) na testovací sadě.


In [ ]:
# Krok 1 & 2: Import knihoven a načtení dat
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Načtení datasetu automobilů
try:
    cars_df = pd.read_csv('car_data.csv')
except FileNotFoundError:
    try:
        cars_df = pd.read_csv('data/car_data.csv')
    except FileNotFoundError:
        cars_df = pd.read_csv('data/MAL_downloadable materials_session 2/Homework/car_data.csv')

print(f"Dataset úspěšně načten! Tvar: {cars_df.shape}")
cars_df.head()


### 📊 Krok 3: Souhrnný report dat (ydata_profiling)
Zadání explicitně požaduje generování přehledového reportu pomocí `ydata_profiling` (dříve `pandas_profiling`).  
*Poznámka:* V Google Colab spustíte instalaci pomocí `!pip install ydata_profiling`. V produkčních dashboardech či offline prostředích lze report vygenerovat jako HTML widget.


In [ ]:
# Krok 3: Generování profilovacího reportu (dle zadání)
# !pip install ydata_profiling

try:
    import ydata_profiling as ydp
    # cars_report = ydp.ProfileReport(cars_df, title="Car Data Profiling Report", minimal=True)
    # cars_report.to_file("cars_report.html")
    print("Knihovna ydata_profiling je importována. Report lze zobrazit přes cars_report.")
except ImportError:
    print("Tip pro Colab / Jupyter: Nainstalujte knihovnu příkazem '!pip install ydata_profiling'")


### 🧹 Krok 4: Čištění dat & náprava kategorických defektů
Při detailním průzkumu dat odhalíme několik zásadních defektů:
1. **Technické indexy a identifikátory:** Sloupce `Unnamed: 0` a `id` nenesou žádnou prediktivní hodnotu o technickém stavu vozu.
2. **Kardinalita modelu:** Sloupec `model` obsahuje 1 580 unikátních hodnot. One-hot kódování modelu by vedlo k extrémní řídkosti matice (1 689 sloupců) a přeučení. V souladu se zadáním jej odstraňujeme jako zbytečný atribut s nadměrnou kardinalitou ve prospěch robustnější značky (`manufacturer`).
3. **Excel Date Auto-Format Bug:** Sloupec `doors` obsahuje hodnoty `'04-May'` (4-5 dveří) a `'02-Mar'` (2-3 dveře), způsobené automatickým formátováním data v programu Microsoft Excel při exportu do CSV. Mapujeme je zpět na `'4-5'` a `'2-3'`.
4. **Scraper / Typo Outlier:** V datech se nachází extrémní anomálie – užitkový vůz OPEL Combo z roku 1999 s cenou **26 307 500 USD**! Ponechání tohoto jediného chybného inzerátu by vedlo k masivnímu zkreslení průměrů v listových uzlech regresního stromu a zápornému $R^2$. Filtrujeme inzeráty s cenou nad 20 mil. USD.


In [ ]:
# Krok 4: Odstranění zbytečných sloupců a oprava dat
# Odstranění prázdných řádků (10 chybně uvozovkovaných řádků v CSV)
df_clean = cars_df.dropna(subset=['price']).copy()

# 1. Odstranění technických a vysoko-kardinálních sloupců
drop_cols = ['Unnamed: 0', 'id', 'model']
df_clean = df_clean.drop(columns=[c for c in drop_cols if c in df_clean.columns])

# 2. Oprava sloupce doors (Excel date auto-formatting)
door_mapping = {'04-May': '4-5', '02-Mar': '2-3', '>5': '>5'}
df_clean['doors'] = df_clean['doors'].replace(door_mapping)

# 3. Ošetření extrémní anomálie (26M USD Opel Combo)
df_clean = df_clean[df_clean['price'] < 20000000].copy()

print(f"Očištěný tvar datasetu: {df_clean.shape}")
print("Unikátní hodnoty doors po opravě:", df_clean['doors'].value_counts().to_dict())
df_clean.describe()


### 🔢 Krok 5: One-Hot kódování kategorických proměnných
Kategorické proměnné (`manufacturer`, `category`, `leather_interior`, `fuel_type`, `gear_box_type`, `drive_wheels`, `doors`, `wheel`, `color`) transformujeme pomocí `pd.get_dummies(..., drop_first=True)` na binární indikátory srozumitelné pro model náhodného lesa.


In [ ]:
# Krok 5: One-Hot Encoding
cat_cols = [
    'manufacturer', 'category', 'leather_interior', 'fuel_type',
    'gear_box_type', 'drive_wheels', 'doors', 'wheel', 'color'
]

df_encoded = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True)
feature_names = [c for c in df_encoded.columns if c != 'price']

X = df_encoded[feature_names]
y = df_clean['price']

print(f"Počet příznaků po One-Hot kódování: {X.shape[1]}")
X.head(3)


### ✂️ Krok 6: Rozdělení na trénovací a testovací sadu (70:30)
Data rozdělíme s fixním `random_state=42` v poměru 70 % trénovací množina a 30 % testovací množina.


In [ ]:
# Krok 6: Train-Test Split (70:30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Trénovací sada: {X_train.shape[0]} vzorků, {X_train.shape[1]} příznaků")
print(f"Testovací sada : {X_test.shape[0]} vzorků, {X_test.shape[1]} příznaků")


### 🌲 Kroky 7, 8 & 9: Instance RandomForestRegressor a optimalizace hyperparametrů
Inicializujeme `RandomForestRegressor(random_state=42, n_jobs=-1)` a definujeme mřížku parametrů `params`:
- `max_depth`: [10, 15, 20]
- `min_samples_leaf`: [1, 2, 4]
- `n_estimators`: [50, 100, 150]

Pro optimalizaci použijeme `GridSearchCV` s metrikou `scoring='neg_mean_squared_error'`, což přímo odpovídá požadavku na minimalizaci **Mean Squared Error (MSE)**.


In [ ]:
# Kroky 7, 8 & 9: Instance modelu, mřížka parametrů a GridSearchCV
rf_regressor = RandomForestRegressor(random_state=42, n_jobs=-1)

params = {
    'max_depth': [10, 15, 20],
    'min_samples_leaf': [1, 2, 4],
    'n_estimators': [50, 100, 150]
}

grid_search = GridSearchCV(
    estimator=rf_regressor,
    param_grid=params,
    scoring='neg_mean_squared_error',
    cv=3,
    n_jobs=-1,
    return_train_score=True
)

print("Spouštím optimalizaci hyperparametrů (GridSearchCV přes MSE)...")
grid_search.fit(X_train, y_train)

print(f"Nejlepší nalezené parametry: {grid_search.best_params_}")
best_rmse_cv = np.sqrt(-grid_search.best_score_)
print(f"Nejlepší CV RMSE: {best_rmse_cv:.2f} USD")


### 🎯 Kroky 10 & 11: Trénování nejlepšího modelu a predikce na testovací sadě
Objekt `grid_search.best_estimator_` již obsahuje natrénovaný model s optimální kombinací hyperparametrů. Následně vygenerujeme predikce pro neviděná testovací data `X_test`.


In [ ]:
# Kroky 10 & 11: Nejlepší model a testovací predikce
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print(f"Vygenerováno {len(y_pred)} predikcí cen vozů na testovací sadě.")


### 📏 Krok 12: Vyhodnocení testovacích metrik (MAE, RMSE, R²)
Dle zadání počítáme:
1. **MAE (Mean Absolute Error):** Průměrná absolutní odchylka odhadované ceny od reálné prodejní ceny.
2. **RMSE (Root Mean Squared Error):** Odmocnina průměrné čtvercové chyby penalizující velké odchylky.
3. **$R^2$ (Koeficient determinace):** Procento variability cen vysvětlené modelem náhodného lesa.


In [ ]:
# Krok 12: Výpočet metrik na testovací množině
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("=" * 45)
print("📊 VÝSLEDKY MODELU NÁHODNÉHO LESA NA TESTOVACÍ SADĚ")
print("=" * 45)
print(f"Průměrná absolutní chyba (MAE) : {mae:,.2f} USD")
print(f"Průměrná čtvercová chyba (MSE) : {mse:,.2f}")
print(f"Odmocnina MSE (RMSE)           : {rmse:,.2f} USD")
print(f"Koeficient determinace (R²)    : {r2:.4f} ({r2*100:.2f} % vysvětlené variance)")
print("=" * 45)


### 💡 Analýza důležitosti příznaků (Feature Importance)
Které vlastnosti automobilu mají největší vliv na predikovanou cenu?


In [ ]:
# Top 15 nejdůležitějších příznaků
importances = pd.Series(best_model.feature_importances_, index=feature_names).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.head(15).values, y=importances.head(15).index, palette='viridis')
plt.title('Top 15 nejdůležitějších příznaků pro predikci ceny automobilu')
plt.xlabel('Gini Importance (MDI)')
plt.ylabel('Příznak')
plt.tight_layout()
plt.show()
